In [ ]:
from imblearn.over_sampling import SMOTE
import cv2
import os
import sys
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, mean_squared_error, mean_absolute_error
from skimage.feature import hog
from skimage.color import rgb2gray
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np

def extract_hog_features(image):
    """Extract HOG features from a single image."""
    # image = skimage.transform.resize(image.numpy(), (64, 128), anti_aliasing=True)
    gray_img = rgb2gray(image)
    features = hog(
        gray_img,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        block_norm='L2-Hys',
        visualize=False
    )
    print('Shape: ', features.shape)

    return features

old = []

def load_images(image_dir, data):
    labels = []
    images = []
    gt_stats = []
    dirs = [image_dir + '/CS', image_dir + '/Healthy']
    for idx, image_dir in enumerate(dirs):
        for filename in sorted(os.listdir(image_dir)):
            if filename.endswith('.png') or filename.endswith('.jpg'):

                # Decode filename to extract sequence number, gender, and age
                # print(filename)
                print(filename)
                # seq_number = int(filename[:-4]) 
                seq_number = filename[:-4]
                if seq_number.endswith("a"):
                    seq_number = seq_number[:-1]  # Remove the last character if it i
                if seq_number.endswith("a"):
                    seq_number = seq_number[:-1]  # Remove the last character if it i
                print(seq_number)
                seq_number = int(seq_number)
                # Find the corresponding row in the dataset
                row = data[data['pic_id'] == seq_number]

                if row.empty:
                    print(f"No matching row found for {filename}")
                    sys.exit(0)
                    continue

                # Drop the Disease Classification column to use all other columns as label
                label_row = row.drop(columns=['pic_id']).iloc[0]
                # 
                gt_stats.append(label_row.values)

                img_path = os.path.join(image_dir, filename)
                image = cv2.imread(img_path)
                image = cv2.resize(image, (224, 224))

                labels.append(idx)
                images.append(image)
    
    
    return np.array(gt_stats), np.array(labels), np.array(images)/1.0  # Normalize images

# File paths to the dataset and image directories
# file_path = '/Users/srivatsavkannan/Datasets/C-Spine Xray/X-ray Atlas/results.xlsx'
file_path = '/Users/srivatsavkannan/Datasets/CervicalNew10/results_ros.xlsx'
data = pd.read_excel(file_path, header=0).dropna()

# train_image_dir = '/Users/srivatsavkannan/Datasets/FinalCervicalDataset/Train_comp+normal_half'
# val_image_dir = '/Users/srivatsavkannan/Datasets/FinalCervicalDataset/Val_comp+normal_half'

train_image_dir = '/Users/srivatsavkannan/Datasets/CervicalNew10/TrainROSCroppedAugNormalizedCLAHE'
val_image_dir = '/Users/srivatsavkannan/Datasets/CervicalNew10/ValROSCroppedCLAHE'

# Load training and validation images and labels
X_train, y_train, images_train = load_images(train_image_dir, data)
X_val, y_val, images_val = load_images(val_image_dir, data)
hog_train = []
hog_val = []
for x in images_train:
    hog_train.append(extract_hog_features(x))
    
for x in images_val:
    hog_val.append(extract_hog_features(x))



hog_train = np.array(hog_train)
hog_val = np.array(hog_val)

print(images_train[0])
print(X_train.shape)
print(X_val.shape)
print(images_train.shape)
print(hog_train.shape)

print(y_train.shape)
print(y_val.shape)
print(images_val.shape)
print(hog_val.shape)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

def build_saint_with_efficientnet_model(tabular_input_dim, image_input_shape):
    # Tabular Data Processing (SAINT-style with Multi-Head Self-Attention)
    tabular_input = layers.Input(shape=(tabular_input_dim,), name="tabular_input")
    x = layers.Reshape((1, tabular_input_dim))(tabular_input)  # Convert to (batch, 1, features) for attention

    num_heads = 4
    key_dim = 32

    for _ in range(2):  
        residual = x
        x = layers.MultiHeadAttention(num_heads=num_heads, key_dim=key_dim)(x, x)
        x = layers.Add()([x, residual])  # Skip connection
        x = layers.LayerNormalization()(x)  # Normalization after attention

    x = layers.Flatten()(x)  # Flatten before merging with other modalities
    # Image data input branch
    image_input = layers.Input(shape=image_input_shape, name="image_input")
    base_model = tf.keras.applications.EfficientNetB7(
        include_top=False,
        weights='imagenet',
        input_shape=image_input_shape
    )   
    base_model.trainable = False  # Freeze the base model

    # Pass inputs through the base model
    y = base_model(image_input, training=False)
    y = tf.keras.layers.GlobalAveragePooling2D()(y)  # Global average pooling for 4D to 2D
    y = tf.keras.layers.Dense(512, activation='relu')(y)

    # Combine tabular and image features (Late Fusion)
    combined = layers.Concatenate()([x, y])
    combined = layers.Dense(128, activation='relu')(combined)
    combined = layers.Dropout(0.3)(combined)

    # Output layer (Binary classification)
    outputs = layers.Dense(1, activation='sigmoid', name="output")(combined)

    # Create the model
    model = models.Model(inputs=[tabular_input, image_input], outputs=outputs)
    return model

def build_saint_with_vgg19_model(tabular_input_dim, image_input_shape):
    # Tabular Data Processing (SAINT-style with Multi-Head Self-Attention)
    tabular_input = layers.Input(shape=(tabular_input_dim,), name="tabular_input")
    x = layers.Reshape((1, tabular_input_dim))(tabular_input)  # Convert to (batch, 1, features) for attention

    num_heads = 4
    key_dim = 32

    for _ in range(2):  
        residual = x
        x = layers.MultiHeadAttention(num_heads=num_heads, key_dim=key_dim)(x, x)
        x = layers.Add()([x, residual])  # Skip connection
        x = layers.LayerNormalization()(x)  # Normalization after attention

    x = layers.Flatten()(x)  # Flatten before merging with other modalities
    # Image data input branch
    image_input = layers.Input(shape=image_input_shape, name="image_input")
    base_model = tf.keras.applications.VGG19(
        include_top=False,
        weights='imagenet',
        input_shape=image_input_shape
    )   
    base_model.trainable = False  # Freeze the base model

    # Pass inputs through the base model
    y = base_model(image_input, training=False)
    y = tf.keras.layers.GlobalAveragePooling2D()(y)  # Global average pooling for 4D to 2D
    y = tf.keras.layers.Dense(512, activation='relu')(y)

    # Combine tabular and image features (Late Fusion)
    combined = layers.Concatenate()([x, y])
    combined = layers.Dense(128, activation='relu')(combined)
    combined = layers.Dropout(0.3)(combined)

    # Output layer (Binary classification)
    outputs = layers.Dense(1, activation='sigmoid', name="output")(combined)

    # Create the model
    model = models.Model(inputs=[tabular_input, image_input], outputs=outputs)
    return model

def build_saint_with_resnet50_model(tabular_input_dim, image_input_shape):
    # Tabular Data Processing (SAINT-style with Multi-Head Self-Attention)
    tabular_input = layers.Input(shape=(tabular_input_dim,), name="tabular_input")
    x = layers.Reshape((1, tabular_input_dim))(tabular_input)  # Convert to (batch, 1, features) for attention

    num_heads = 4
    key_dim = 32

    for _ in range(2):  
        residual = x
        x = layers.MultiHeadAttention(num_heads=num_heads, key_dim=key_dim)(x, x)
        x = layers.Add()([x, residual])  # Skip connection
        x = layers.LayerNormalization()(x)  # Normalization after attention

    x = layers.Flatten()(x)  # Flatten before merging with other modalities
    # Image data input branch
    image_input = layers.Input(shape=image_input_shape, name="image_input")
    base_model = tf.keras.applications.VGG19(
        include_top=False,
        weights='imagenet',
        input_shape=image_input_shape
    )   
    base_model.trainable = False  # Freeze the base model

    # Pass inputs through the base model
    y = base_model(image_input, training=False)
    y = tf.keras.layers.GlobalAveragePooling2D()(y)  # Global average pooling for 4D to 2D
    y = tf.keras.layers.Dense(512, activation='relu')(y)

    # Combine tabular and image features (Late Fusion)
    combined = layers.Concatenate()([x, y])
    combined = layers.Dense(128, activation='relu')(combined)
    combined = layers.Dropout(0.3)(combined)

    # Output layer (Binary classification)
    outputs = layers.Dense(1, activation='sigmoid', name="output")(combined)

    # Create the model
    model = models.Model(inputs=[tabular_input, image_input], outputs=outputs)
    return model

def build_saint_with_densenet121_model(tabular_input_dim, image_input_shape):
    # Tabular Data Processing (SAINT-style with Multi-Head Self-Attention)
    tabular_input = layers.Input(shape=(tabular_input_dim,), name="tabular_input")
    x = layers.Reshape((1, tabular_input_dim))(tabular_input)  # Convert to (batch, 1, features) for attention

    num_heads = 4
    key_dim = 32

    for _ in range(2):  
        residual = x
        x = layers.MultiHeadAttention(num_heads=num_heads, key_dim=key_dim)(x, x)
        x = layers.Add()([x, residual])  # Skip connection
        x = layers.LayerNormalization()(x)  # Normalization after attention

    x = layers.Flatten()(x)  # Flatten before merging with other modalities
    # Image data input branch
    image_input = layers.Input(shape=image_input_shape, name="image_input")
    base_model = tf.keras.applications.VGG19(
        include_top=False,
        weights='imagenet',
        input_shape=image_input_shape
    )   
    base_model.trainable = False  # Freeze the base model

    # Pass inputs through the base model
    y = base_model(image_input, training=False)
    y = tf.keras.layers.GlobalAveragePooling2D()(y)  # Global average pooling for 4D to 2D
    y = tf.keras.layers.Dense(512, activation='relu')(y)

    # Combine tabular and image features (Late Fusion)
    combined = layers.Concatenate()([x, y])
    combined = layers.Dense(128, activation='relu')(combined)
    combined = layers.Dropout(0.3)(combined)

    # Output layer (Binary classification)
    outputs = layers.Dense(1, activation='sigmoid', name="output")(combined)

    # Create the model
    model = models.Model(inputs=[tabular_input, image_input], outputs=outputs)
    return model

def build_efficientnet_model(image_input_shape):
    # Image data input branch
    image_input = layers.Input(shape=image_input_shape, name="image_input")
    base_model = tf.keras.applications.EfficientNetB7(
        include_top=False,
        weights='imagenet',
        input_shape=image_input_shape
    )   
    base_model.trainable = False  # Freeze the base model

    # Pass inputs through the base model
    y = base_model(image_input, training=False)
    y = tf.keras.layers.GlobalAveragePooling2D()(y)  # Global average pooling for 4D to 2D
    y = tf.keras.layers.Dense(512, activation='relu')(y)

    # Fully connected layers
    combined = layers.Dense(128, activation='relu')(y)
    combined = layers.Dropout(0.3)(combined)

    # Output layer (Binary classification)
    outputs = layers.Dense(1, activation='sigmoid', name="output")(combined)

    # Create the model
    model = models.Model(inputs=image_input, outputs=outputs)
    return model

def build_saint_model(tabular_input_dim):
    # Tabular Data Processing (SAINT-style with Multi-Head Self-Attention)
    tabular_input = layers.Input(shape=(tabular_input_dim,), name="tabular_input")
    x = layers.Reshape((1, tabular_input_dim))(tabular_input)  # Convert to (batch, 1, features) for attention

    num_heads = 4
    key_dim = 32

    for _ in range(2):  # Two transformer layers
        residual = x
        x = layers.MultiHeadAttention(num_heads=num_heads, key_dim=key_dim)(x, x)
        x = layers.Add()([x, residual])  # Skip connection
        x = layers.LayerNormalization()(x)  # Normalization after attention

    x = layers.Flatten()(x)  # Flatten to prepare for output
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)

    # Output layer (Binary classification)
    outputs = layers.Dense(1, activation='sigmoid', name="output")(x)

    # Create the model
    model = models.Model(inputs=tabular_input, outputs=outputs)
    return model



In [ ]:
input_dim = X_train.shape[1]
image_input_dim = (224,224,3)
hog_input_dim = hog_train[0].shape[0]
# print(type(hog_input_dim))
# print(type(input_dim))
model = build_saint_model(input_dim)

X_train = tf.convert_to_tensor(X_train, dtype=tf.float32)
images_train = tf.convert_to_tensor(images_train, dtype=tf.float32)
hog_train = tf.convert_to_tensor(hog_train, dtype=tf.float32)
y_train = tf.convert_to_tensor(y_train, dtype=tf.float32)

X_val = tf.convert_to_tensor(X_val, dtype=tf.float32)
images_val = tf.convert_to_tensor(images_val, dtype=tf.float32)
hog_val = tf.convert_to_tensor(hog_val, dtype=tf.float32)
y_val = tf.convert_to_tensor(y_val, dtype=tf.float32)

model.compile(optimizer=tf.keras.optimizers.Adam(),
              loss='binary_crossentropy',
              metrics=['accuracy'])

# Train the model with class weights
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=100, restore_best_weights=True)
history = model.fit([X_train], y_train,
                    validation_data=([X_val], y_val),
                    epochs=300,
                    callbacks=[early_stop],
                    batch_size=32,
                    verbose=1)


model.save("effnet_7030_30epochs_cervicalnew10.keras")




In [ ]:
from sklearn.metrics import confusion_matrix
from matplotlib import pyplot as plt
import seaborn as sns

model = tf.keras.models.load_model("quant_7030_30epochs_cervicalnew10.keras")
model.summary()
# train_loss = history.history['loss']
# val_loss = history.history['val_loss']
# train_acc = history.history['accuracy']
# val_acc = history.history['val_accuracy']
# epochs = range(1, len(train_loss) + 1)
# # 
import json
# # # 
# # # Save history
# with open("effnet_quant_7030_30epochs_cervicalnew10.json", "w") as f:
#     json.dump(history.history, f)
# # # 
# Load history later
# with open("effnet_quant_7030_30epochs_cervicalnew10.json", "r") as f:
#     loaded_history = json.load(f)
# 
# 
# # Access data
# train_loss = loaded_history['loss']
# val_loss = loaded_history['val_loss']
# train_acc = loaded_history['accuracy']
# val_acc = loaded_history['val_accuracy']
# epochs = range(1, len(train_loss) + 1)
# 
# # Function to compute moving average
# def moving_average(data, window_size=5):
#     return pd.Series(data).rolling(window=window_size, min_periods=1).mean()
# 
# # Apply moving average smoothing
# window_size = 5  # Adjust this value for more or less smoothing
# smoothed_train_loss = moving_average(train_loss, window_size)
# smoothed_val_loss = moving_average(val_loss, window_size)
# smoothed_train_acc = moving_average(train_acc, window_size)
# smoothed_val_acc = moving_average(val_acc, window_size)
# 
# # Plot smoothed loss
# plt.figure(figsize=(12, 5))
# 
# # --- Training & Validation Loss ---
# plt.subplot(1, 2, 1)
# # Unsmoothed (lighter, transparent)
# plt.plot(epochs, train_loss, color='lightskyblue', alpha=0.5, label='Training loss')
# plt.plot(epochs, val_loss, color='lightcoral', alpha=0.5, label='Validation loss')
# 
# # Smoothed (darker, on top)
# plt.plot(epochs, smoothed_train_loss, color='royalblue', linewidth=2, label='Training loss (smoothed)')
# plt.plot(epochs, smoothed_val_loss, color='crimson', linewidth=2, label='Validation loss (smoothed)')
# 
# plt.title('Training and Validation Loss')
# plt.xlabel('Epochs')
# plt.ylabel('Loss')
# plt.legend()
# 
# # --- Training & Validation Accuracy ---
# plt.subplot(1, 2, 2)
# # Unsmoothed (lighter, transparent)
# plt.plot(epochs, train_acc, color='lightskyblue', alpha=0.5, label='Training accuracy')
# plt.plot(epochs, val_acc, color='lightcoral', alpha=0.5, label='Validation accuracy')
# 
# # Smoothed (darker, on top)
# plt.plot(epochs, smoothed_train_acc, color='royalblue', linewidth=2, label='Training accuracy (smoothed)')
# plt.plot(epochs, smoothed_val_acc, color='crimson', linewidth=2, label='Validation accuracy (smoothed)')
# 
# plt.title('Training and Validation Accuracy')
# plt.xlabel('Epochs')
# plt.ylabel('Accuracy')
# plt.legend()

# plt.show()


# Predictions and metrics
# y_pred_train = (model.predict([X_train, images_train]) > 0.5).astype(int)
y_pred_test = (model.predict([X_val]) > 0.5).astype(int)
# y_pred_test_proba = model.predict([X_val, images_val, hog_val]).flatten()

# Compute metrics
# print("Training Classification Report:\n", classification_report(y_train, y_pred_train, digits=4))
class_names = ["CS", "Healthy"]
print("Testing Classification Report:\n", classification_report(y_val, y_pred_test, digits=4))
cm = confusion_matrix(y_val, y_pred_test)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names, 
            annot_kws={"size": 16, "weight": "bold"})  # Making numbers bigger and bolder
plt.xlabel('Predicted', fontsize=14, fontweight="bold")
plt.ylabel('True', fontsize=14, fontweight="bold")
plt.title('Confusion Matrix', fontsize=16, fontweight="bold")
plt.show()



In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score
# 
# model = tf.keras.models.load_model("quant_effnet_ros_aug_norm_clahe.keras")
# y_pred_test = (model.predict([X_val, images_val]) > 0.5).astype(int)
cm = confusion_matrix(y_val, y_pred_test)
# Compute confusion matrix
tp, fn, fp, tn = confusion_matrix(y_val, y_pred_test).ravel()
print(tp, fn, fp, tn)
# Sensitivity (Recall for Disease Present)
sensitivity = tp / (tp + fn)  # Same as recall for class 0

# Specificity (Recall for Disease Absent)
specificity = tn / (tn + fp)

# Precision (PPV - Positive Predictive Value)
precision = tp / (tp + fp)

# Negative Predictive Value (NPV)
npv = tn / (tn + fn)

# F1-Score for the Positive Class
f1_score_pos = 2 * (precision * sensitivity) / (precision + sensitivity)

# Overall Accuracy
accuracy = accuracy_score(y_val, y_pred_test)

# AUC-ROC Score
auc_roc = roc_auc_score(y_val, y_pred_test)

# Print results
print(f"Binary Classification Metrics:")
print(f"Sensitivity (Recall for Disease Present)  : {sensitivity:.4f}")
print(f"Specificity (Recall for Disease Absent)  : {specificity:.4f}")
print(f"Precision (PPV)                           : {precision:.4f}")
print(f"Negative Predictive Value (NPV)           : {npv:.4f}")
print(f"F1-Score for Disease Present (Class 0)   : {f1_score_pos:.4f}")
print(f"Overall Accuracy                         : {accuracy:.4f}")
print(f"AUC-ROC Score                            : {auc_roc:.4f}")


In [ ]:
import os
import sys
import cv2
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import pandas as pd

# Load images from directory and match them with labels from dataset
def load_images(image_dir, data):
    labels = []
    images = []
    gt_stats = []
    filenames = []

    dirs = [os.path.join(image_dir, 'CS'), os.path.join(image_dir, 'Healthy')]
    
    for idx, image_dir in enumerate(dirs):
        for filename in sorted(os.listdir(image_dir)):
            if filename.endswith('.png') or filename.endswith('.jpg'):
                print(f"Processing: {filename}")
                seq_number = filename[:-4]  # Extract sequence ID from filename
                
                # Find the corresponding row in the dataset
                row = data[data['pic_id'] == seq_number]
                
                if row.empty:
                    print(f"No matching row found for {filename}")
                    sys.exit(0)
                
                # Drop 'pic_id' to use other columns as ground truth labels
                label_row = row.drop(columns=['pic_id']).iloc[0]
                gt_stats.append(label_row.values)

                img_path = os.path.join(image_dir, filename)
                image = cv2.imread(img_path)
                image = cv2.resize(image, (224, 224))

                labels.append(idx)  # Binary label based on folder structure
                images.append(image)
                filenames.append(filename)

    return np.array(gt_stats), np.array(labels), np.array(images) / 1.0, filenames  # Normalize images

# file_path = '/Users/srivatsavkannan/Datasets/C-Spine Xray/X-ray Atlas/results.xlsx'
csv_path = '/Users/srivatsavkannan/Datasets/CervicalNew10/results_ros.xlsx'
data = pd.read_excel(file_path, header=0).dropna()

# train_image_dir = '/Users/srivatsavkannan/Datasets/FinalCervicalDataset/Train_comp+normal_half'
# val_image_dir = '/Users/srivatsavkannan/Datasets/FinalCervicalDataset/Val_comp+normal_half'

train_image_dir = '/Users/srivatsavkannan/Datasets/CervicalNew10/TrainROSCroppedAugNormalizedCLAHE'
val_image_dir = '/Users/srivatsavkannan/Datasets/CervicalNew10/ValROSCroppedCLAHE'

data = pd.read_excel(csv_path, dtype={'pic_id': str})  # Ensure pic_id is treated as string
gt_stats, y_val, images_val, filenames = load_images(val_image_dir, data)

# Load trained model
# model = tf.keras.models.load_model("quant_effnet_ros_aug_norm_clahe.keras")

# Predict labels one by one
fig, axes = plt.subplots(2, 5, figsize=(20, 8))  # Adjust grid size as needed
axes = axes.ravel()

error_count = 0

for i in range(len(images_val)):
    image = np.expand_dims(images_val[i], axis=0)  # Add batch dimension
    prediction = (model.predict([np.expand_dims(gt_stats[i], axis=0), image]) > 0.5).astype(int)

    if prediction != y_val[i]:  # Misclassified
        ax = axes[error_count]
        ax.imshow(images_val[i], cmap='gray')
        ax.set_title(f"Wrong: {filenames[i]}\nPred: {prediction[0][0]}, GT: {y_val[i]}")
        ax.axis("off")

        error_count += 1
        if error_count >= 10:  # Stop after plotting 10 misclassified images
            break

plt.tight_layout()
plt.show()
